In [1]:
"""
A full tutorial is available here:
https://nichecompass.readthedocs.io/en/latest/tutorials/notebooks/mouse_cns_spatial_reference_mapping.html

Here we illustrate preaparation of our core dataset as a reference for mapping the validation data
"""

'\nA full tutorial is available here:\nhttps://nichecompass.readthedocs.io/en/latest/tutorials/notebooks/mouse_cns_spatial_reference_mapping.html\n\nHere we illustrate preaparation of our core dataset as a reference for mapping the validation data\n'

In [2]:
import os, sys
import random
import warnings
import logging
from datetime import datetime
# import gdown
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns
import squidpy as sq
#from matplotlib import gridspec
#from sklearn.preprocessing import MinMaxScaler
from re import sub
import numpy as np
import pickle

# from nichecompass.models import NicheCompass
# from nichecompass.utils import (add_gps_from_gp_dict_to_adata,
#                                 create_new_color_dict,
#                                 compute_communication_gp_network,
#                                 visualize_communication_gp_network,
#                                 extract_gp_dict_from_mebocost_ms_interactions,
#                                 #extract_gp_dict_from_mebocost_es_interactions,
#                                 extract_gp_dict_from_nichenet_lrt_interactions,
#                                 extract_gp_dict_from_omnipath_lr_interactions,
#                                 #filter_and_combine_gp_dict_gps,
#                                 filter_and_combine_gp_dict_gps_v2,
#                                 generate_enriched_gp_info_plots)


# %%


/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: 

# Set output dir + where to find niche compass files

In [3]:
"""
make sure inside this path, you have the folders gene_annotations 
and gene_programs with the files
(available from https://github.com/Lotfollahi-lab/nichecompass/tree/main/data)
"""

handle='/lustre/scratch124/cellgen/haniffa/projects/developmental_fibroblasts/nobackup_output/nichecompasss/nichecompass/' 


# Choose number of SVGs to reduce dataset to

In [4]:
n_svg=1024


# Set up reference and query (important part)

In [5]:
"""
load adata (includes reference and query)
- note that sample id is in adata.obs["Sample"]
- cell type is in adata.obs["Annotation"]

if using our adata as reference, then either:
1. remove all query samples (in query_batches below), or
2. add query samples to reference_batches, 

and then add your sample id's to query_batches
"""

#'/nfs/team298/ls34/xenium_atlas/model_ALL_CLEAN_scanvi_ALL/adata_counts_integrated_final_colored.h5ad'


ADATA_PATH = '/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/adata_nemo_all2.h5ad'
def remove_markers(LIST):
    try: 
        LIST = {key: [gene for gene in genes if gene in adata_5k.var_names] 
                           for key, genes in LIST.items()}
    except: 
        LIST =[gene for gene in LIST if gene in adata_5k.var_names]
    return LIST


adata_vis=sc.read_h5ad(ADATA_PATH)  





In [6]:
adata_vis.layers["counts"] = adata_vis.X.copy()

In [7]:
adata_vis.shape

(1851985, 4952)

In [8]:
adata_vis.obs["info_id6"].value_counts()

info_id6
3D_BK25_week12-D2                                  41317
BK39_Week 12                                       32052
BK30_Day 14                                        30413
BK44-SKI-27-FO-1-S4-E2                             28549
BK40-SKI-27-FO-1-S4-D2                             28305
                                                   ...  
Baseline_resolved_CE6-SKI-20-FO-1-S22-C2            6030
Baseline_resolved_CE6-SKI-20-FO-1-S22_replicate     5846
Baseline_resolved_CE4-SKI-27-FO-1-S22-B2            5641
Baseline_resolved_CE4-SKI-27-FO-1-S22_replicate     5474
BK20_Lesional Baseline                              2290
Name: count, Length: 122, dtype: int64

In [9]:
adata_vis.obs["sample"]=adata_vis.obs["info_id6"]
adata_vis.obs["sample"].value_counts()

sample
3D_BK25_week12-D2                                  41317
BK39_Week 12                                       32052
BK30_Day 14                                        30413
BK44-SKI-27-FO-1-S4-E2                             28549
BK40-SKI-27-FO-1-S4-D2                             28305
                                                   ...  
Baseline_resolved_CE6-SKI-20-FO-1-S22-C2            6030
Baseline_resolved_CE6-SKI-20-FO-1-S22_replicate     5846
Baseline_resolved_CE4-SKI-27-FO-1-S22-B2            5641
Baseline_resolved_CE4-SKI-27-FO-1-S22_replicate     5474
BK20_Lesional Baseline                              2290
Name: count, Length: 122, dtype: int64

In [10]:
adata_vis

AnnData object with n_obs × n_vars = 1851985 × 4952
    obs: 'sample_id', 'barcode', 'GSE', 'Site_status', 'Patient_status', 'Location', 'Age', 'Sex', 'n_genes', 'dataset_id', 'n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'QC_hi', 'QC_mid', 'lvl5_annotation', 'Mapping_status', 'scanvi_predictions', 'lvl5_annotation_new', 'lvl5_annotation_new2', 'lvl5_annotation_new3', 'lvl5_annotation_new_archive', 'lvl5_annotation_new_preoprhan', 'lvl5_annotation_new10', 'lvl5_annotation_new11', 'test', 'test_n', 'lvl5_annotation_new12', 'lvl5_annotation_new13', 'lvl4_annotation', 'lvl0', 'temp', 't', 'leiden_res0.1', 'Site_status_simple', 'StatusMilo', 'atlas_status', 'atlas_status_reynolds', 'atlas_status_reynolds_simple', 'atlas_status_simple', 'atlas_status_simple2', 'Site_status_binary', 'scanvi_labels', 'cell

In [11]:
adata_vis.obs["Annotation"]=adata_vis.obs["provisional2"]

In [12]:
# adata_vis.obs["Annotation"].value_counts()

In [13]:
adata_vis.obs["sample"].unique().to_list()

/tmp/ipykernel_2842908/3193344545.py:1: FutureWarning: Categorical.to_list is deprecated and will be removed in a future version. Use obj.tolist() instead
  adata_vis.obs["sample"].unique().to_list()


['BK36-SKI-27-FO-1-S4-D1',
 'BK36-SKI-27-FO-1-S5-E1',
 'BK37-SKI-27-FO-1-S4-C1',
 'BK40-SKI-27-FO-1-S4-D2',
 'BK42-SKI-27-FO-1-S4-C2',
 'BK44-SKI-27-FO-1-S4-E2',
 'BK58-SKI-27-FO-1-S5-A1',
 'BK58-SKI-27-FO-4-S4-A2',
 'BK65-SKI-21-FO-1-S4-B1',
 'BK68-SKI-27-FO-1-S4-B2',
 'Week 8 (resolved)_CE3-SKI-28-FO-4-S22-E1_a',
 'Lesional_CE5-SKI-28-FO-1-S22_replicate',
 'Week 8 (resolved)_CE3-SKI-28-FO-4-S22-E1_b',
 'BK39_Non-lesional Baseline',
 'Baseline_resolved_CE3-SKI-28-FO-1-S22-B1',
 'BK21-SKI-27-FO-1-S8-A3',
 'Week 8 (resolved)_CE5-SKI-27-FO-4-S22-E1',
 'BK23_Lesional Baseline',
 '3D_BK25_week12-D2',
 '3D_BK25_week12-D1orE1b',
 'BK51-SKI-27-FO-2-S9-B2',
 'Baseline_never_CE5-SKI-27-FO-2-S22_replicate',
 '3D_BK22_Lesional_baseline-C2',
 'BK51_Never Lesional',
 'BK20_Week 12',
 'BK30_Lesional Baseline',
 'BK23-SKI-27-FO-1-S8-B1',
 '3D_BK22_Lesional_baseline-A1',
 'Week 8 (resolved)_CE4-SKI-27-FO-3-S22-E2_b',
 'BK21-SKI-27-FO1-S11-C1',
 'BK30_Week 12',
 '3D_BK22_Lesional_baseline-B1',
 'BK27-S

In [14]:
"""
SPLIT INTO REFERENCE AND QUERY
"""
 
"""
replace these query batches with your samples
"""
 

reference_batches = adata_vis.obs["sample"].unique().to_list()
reference_batches

/tmp/ipykernel_2842908/4108310684.py:10: FutureWarning: Categorical.to_list is deprecated and will be removed in a future version. Use obj.tolist() instead
  reference_batches = adata_vis.obs["sample"].unique().to_list()


['BK36-SKI-27-FO-1-S4-D1',
 'BK36-SKI-27-FO-1-S5-E1',
 'BK37-SKI-27-FO-1-S4-C1',
 'BK40-SKI-27-FO-1-S4-D2',
 'BK42-SKI-27-FO-1-S4-C2',
 'BK44-SKI-27-FO-1-S4-E2',
 'BK58-SKI-27-FO-1-S5-A1',
 'BK58-SKI-27-FO-4-S4-A2',
 'BK65-SKI-21-FO-1-S4-B1',
 'BK68-SKI-27-FO-1-S4-B2',
 'Week 8 (resolved)_CE3-SKI-28-FO-4-S22-E1_a',
 'Lesional_CE5-SKI-28-FO-1-S22_replicate',
 'Week 8 (resolved)_CE3-SKI-28-FO-4-S22-E1_b',
 'BK39_Non-lesional Baseline',
 'Baseline_resolved_CE3-SKI-28-FO-1-S22-B1',
 'BK21-SKI-27-FO-1-S8-A3',
 'Week 8 (resolved)_CE5-SKI-27-FO-4-S22-E1',
 'BK23_Lesional Baseline',
 '3D_BK25_week12-D2',
 '3D_BK25_week12-D1orE1b',
 'BK51-SKI-27-FO-2-S9-B2',
 'Baseline_never_CE5-SKI-27-FO-2-S22_replicate',
 '3D_BK22_Lesional_baseline-C2',
 'BK51_Never Lesional',
 'BK20_Week 12',
 'BK30_Lesional Baseline',
 'BK23-SKI-27-FO-1-S8-B1',
 '3D_BK22_Lesional_baseline-A1',
 'Week 8 (resolved)_CE4-SKI-27-FO-3-S22-E2_b',
 'BK21-SKI-27-FO1-S11-C1',
 'BK30_Week 12',
 '3D_BK22_Lesional_baseline-B1',
 'BK27-S

In [15]:
# adata_vis.obs["batch_nc"] = "query"
# adata_vis.obs.loc[adata_vis.obs["sample"].isin(reference_batches), "batch_nc"] = "reference"

In [16]:
# query_check = list(adata_vis[adata_vis.obs["batch_nc"]=="query"].obs["sample"].unique())
# for x in query_batches:
#     if x not in query_check:
#         raise ValueError(f"Batch '{x}' not found in query samples.")

In [17]:
# all_batches = reference_batches +query_batches


In [18]:
### from jimmy lee 
def select_slide2(adata, s, s_col='sample'):
    """ This function selects the data for one slide from the spatial anndata object.
    :param adata: Anndata object with multiple spatial experiments
    :param s: name of selected experiment
    :param s_col: column in adata.obs listing experiment name for each location
    """
    slide = adata[adata.obs[s_col].isin([s]), :]
#     s_keys = list(slide.uns['spatial'].keys())
#     s_spatial = np.array(s_keys)[[s in k for k in s_keys]][0]
#     slide.uns['spatial'] = {s_spatial: slide.uns['spatial'][s_spatial]}
    return slide

In [19]:
spatial_key = "spatial"
n_neighbors = 8
adj_key = "spatial_connectivities"

adata_batch_list = []
print("Processing reference batches...")
for batch in reference_batches:
    print(f"Processing batch {batch}...")
    print("Loading data...")
    adata_batch = select_slide2(adata_vis, batch)
    print(f"Size {adata_batch.shape}")
    print("Computing spatial neighborhood graph...\n")
    # Compute (separate) spatial neighborhood graphs
    logging.info("sq.gr.spatial_neighbors")
    #try:
    sq.gr.spatial_neighbors(adata_batch,
                                coord_type="generic",
                                spatial_key=spatial_key,
                                n_neighs=n_neighbors)
    #except:
    #    continue
    print(f"Spatial neighbours done ## {adata_batch.shape}")

    # Make adjacency matrix symmetric
    adata_batch.obsp[adj_key] = (
        adata_batch.obsp[adj_key].maximum(
            adata_batch.obsp[adj_key].T))
    adata_batch_list.append(adata_batch)
print("List made: ...")
for x in adata_batch_list:
    print(x.shape)
adata_vis = ad.concat(adata_batch_list, join="inner")

Processing reference batches...
Processing batch BK36-SKI-27-FO-1-S4-D1...
Loading data...
Size (17597, 4952)
Computing spatial neighborhood graph...



/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data


Spatial neighbours done ## (17597, 4952)
Processing batch BK36-SKI-27-FO-1-S5-E1...
Loading data...
Size (17263, 4952)
Computing spatial neighborhood graph...



/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data


Spatial neighbours done ## (17263, 4952)
Processing batch BK37-SKI-27-FO-1-S4-C1...
Loading data...
Size (14293, 4952)
Computing spatial neighborhood graph...

Spatial neighbours done ## (14293, 4952)
Processing batch BK40-SKI-27-FO-1-S4-D2...
Loading data...
Size (28305, 4952)
Computing spatial neighborhood graph...

Spatial neighbours done ## (28305, 4952)
Processing batch BK42-SKI-27-FO-1-S4-C2...
Loading data...


...

Spatial neighbours done ## (22120, 4952)
Processing batch BK23_Week 12...
Loading data...
Size (10889, 4952)
Computing spatial neighborhood graph...



/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data


Spatial neighbours done ## (10889, 4952)
Processing batch BK49_Never Lesional...
Loading data...
Size (13463, 4952)
Computing spatial neighborhood graph...



/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data


Spatial neighbours done ## (13463, 4952)
List made: ...
(17597, 4952)
(17263, 4952)
(14293, 4952)
(28305, 4952)
(17785, 4952)
(28549, 4952)
(13223, 4952)
(12678, 4952)
(18107, 4952)
(22344, 4952)
(12797, 4952)
(10880, 4952)
(9921, 4952)
(16373, 4952)
(6191, 4952)
(10598, 4952)
(18081, 4952)
(24391, 4952)
(41317, 4952)
(17172, 4952)
(15951, 4952)
(10686, 4952)
(22025, 4952)
(15447, 4952)
(14893, 4952)
(13445, 4952)
(19000, 4952)
(23006, 4952)
(11677, 4952)
(16359, 4952)
(18667, 4952)
(20434, 4952)
(22488, 4952)
(21439, 4952)
(6030, 4952)
(11032, 4952)
(11401, 4952)
(12499, 4952)
(8068, 4952)
(6127, 4952)
(15084, 4952)
(11260, 4952)
(14252, 4952)
(17717, 4952)
(32052, 4952)
(9151, 4952)
(13422, 4952)
(5641, 4952)
(9223, 4952)
(5846, 4952)
(27362, 4952)
(7620, 4952)
(23770, 4952)
(7907, 4952)
(16626, 4952)
(9154, 4952)
(11636, 4952)
(13995, 4952)
(7708, 4952)
(5474, 4952)
(14662, 4952)
(12612, 4952)
(15715, 4952)
(19416, 4952)
(15242, 4952)
(11921, 4952)
(9786, 4952)
(9713, 4952)
(16500, 

In [20]:
# Combine spatial neighborhood graphs as disconnected components
batch_connectivities = []
len_before_batch = 0
for i in range(len(adata_batch_list)):
    if i == 0: # first batch
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[0].shape[0],
            (adata_vis.shape[0] -
            adata_batch_list[0].shape[0])))
        batch_connectivities.append(sp.hstack(
            (adata_batch_list[0].obsp[adj_key],
            after_batch_connectivities_extension)))
    elif i == (len(adata_batch_list) - 1): # last batch
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_vis.shape[0] -
            adata_batch_list[i].shape[0])))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp[adj_key])))
    else: # middle batches
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0], len_before_batch))
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_vis.shape[0] -
            adata_batch_list[i].shape[0] -
            len_before_batch)))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp[adj_key],
            after_batch_connectivities_extension)))
    len_before_batch += adata_batch_list[i].shape[0]
adata_vis.obsp[adj_key] = sp.vstack(batch_connectivities)



In [21]:
sq.gr.spatial_autocorr(adata_vis, mode="moran", genes=adata_vis.var_names)


In [22]:
sv_genes = adata_vis.uns["moranI"].index[:n_svg].tolist()


In [23]:
adata_vis.var["spatially_variable"] = adata_vis.var_names.isin(sv_genes)
adata_vis.var["keep_gene"] = adata_vis.var["spatially_variable"]
adata_vis = adata_vis[:, adata_vis.var["keep_gene"] == True]
print(f"Keeping {len(adata_vis.var_names)} spatially variable, highly "
       "variable or gene program relevant genes.")


Keeping 1024 spatially variable, highly variable or gene program relevant genes.


In [24]:
import pickle
filepath = f"{handle}/svgenelist.pkl"
with open(filepath, "wb") as f:
    pickle.dump(sv_genes, f)

In [25]:
# sv_genes

In [26]:
# adata_vis

In [27]:
adata_vis.write(ADATA_PATH + ".svg")  
print(ADATA_PATH + ".svg")

/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/adata_nemo_all2.h5ad.svg


In [ ]:
stop

stop

# Split into ref + query adatas

In [28]:
# ADATA_PATH= '/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/adata_all.h5ad.clustered.clustered10.good.prenichecompass.svg'
# adata_vis=sc.read_h5ad(ADATA_PATH)  
# adata_vis.shape

In [29]:
query_batches = list(adata_vis[adata_vis.obs["batch_nc"]=="query"].obs["sample"].unique())
reference_batches = list(adata_vis[adata_vis.obs["batch_nc"]=="reference"].obs["sample"].unique())


In [30]:
adata_vis.obs["batch_nc"].value_counts()

batch_nc
nan      1661841
query     190144
Name: count, dtype: int64

In [31]:
adata_batch_list = []
print("Processing reference batches...")
for batch in reference_batches:
    print(f"Processing batch {batch}...")
    adata_batch = select_slide2(adata_vis, batch)
    sq.gr.spatial_neighbors(adata_batch,
                                coord_type="generic",
                                spatial_key=spatial_key,
                                n_neighs=n_neighbors)
    #except:
    #    continue
    print(f"Spatial neighbours done ## {adata_batch.shape}")

    # Make adjacency matrix symmetric
    adata_batch.obsp[adj_key] = (
        adata_batch.obsp[adj_key].maximum(
            adata_batch.obsp[adj_key].T))
    adata_batch_list.append(adata_batch)
print("List made: ...")
for x in adata_batch_list:
    print(x.shape)
adata_reference = ad.concat(adata_batch_list, join="inner")

Processing reference batches...
List made: ...


ValueError: No objects to concatenate

In [ ]:
print("List made: ...", len(adata_batch_list))
for x in adata_batch_list:
    print(x.shape)

In [ ]:
adata_reference = ad.concat(adata_batch_list, join="inner")

In [ ]:
batch_connectivities = []
len_before_batch = 0
for i in range(len(adata_batch_list)):
    if i == 0: # first batch
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[0].shape[0],
            (adata_reference.shape[0] -
            adata_batch_list[0].shape[0])))
        batch_connectivities.append(sp.hstack(
            (adata_batch_list[0].obsp[adj_key],
            after_batch_connectivities_extension)))
    elif i == (len(adata_batch_list) - 1): # last batch
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_reference.shape[0] -
            adata_batch_list[i].shape[0])))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp[adj_key])))
    else: # middle batches
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0], len_before_batch))
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_reference.shape[0] -
            adata_batch_list[i].shape[0] -
            len_before_batch)))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp[adj_key],
            after_batch_connectivities_extension)))
    len_before_batch += adata_batch_list[i].shape[0]
adata_reference.obsp[adj_key] = sp.vstack(batch_connectivities)


In [ ]:
mapping_entity_key = "mapping_entity"
adata_reference.obs[mapping_entity_key] = "reference"


In [ ]:
adata_reference.write(ADATA_PATH + ".svg.reference")  
print(ADATA_PATH + ".svg.reference")

In [ ]:
adata_vis.obs.batch_nc.value_counts()

In [ ]:
query_batches

In [ ]:
adata_batch_list = []
print("Processing query batches...")
# for batch in query_batches:
#     print(f"Processing batch {batch}...")
#     print("Loading data...")
#     adata_batch = sc.read_h5ad(
#         f"{so_data_folder_path}/{dataset}_{batch}.h5ad")
for batch in query_batches:
   # print(f"Processing batch {batch}...")
    #print("Loading data...")
    adata_batch = select_slide2(adata_vis, batch)
    #print("Computing spatial neighborhood graph...\n")
    # Compute (separate) spatial neighborhood graphs
    sq.gr.spatial_neighbors(adata_batch,
                            coord_type="generic",
                            spatial_key=spatial_key,
                            n_neighs=n_neighbors)
    
    # Make adjacency matrix symmetric
    adata_batch.obsp[adj_key] = (
        adata_batch.obsp[adj_key].maximum(
            adata_batch.obsp[adj_key].T))
    adata_batch_list.append(adata_batch)
adata_query = ad.concat(adata_batch_list, join="inner")

In [ ]:
adata_query.obs["sample"].value_counts()

In [ ]:
len(adata_batch_list)

In [ ]:
# adata_batch_list

In [ ]:
for i in range(len(adata_batch_list)):
    print(adata_batch_list[i].shape)

In [ ]:
# Combine spatial neighborhood graphs as disconnected components
batch_connectivities = []
len_before_batch = 0
for i in range(len(adata_batch_list)):
    if i == 0: # first batch
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[0].shape[0],
            (adata_query.shape[0] -
            adata_batch_list[0].shape[0])))
        batch_connectivities.append(sp.hstack(
            (adata_batch_list[0].obsp[adj_key],
            after_batch_connectivities_extension)))
    elif i == (len(adata_batch_list) - 1): # last batch
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_query.shape[0] -
            adata_batch_list[i].shape[0])))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp[adj_key])))
    else: # middle batches
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0], len_before_batch))
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_query.shape[0] -
            adata_batch_list[i].shape[0] -
            len_before_batch)))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp[adj_key],
            after_batch_connectivities_extension)))
    len_before_batch += adata_batch_list[i].shape[0]
adata_query.obsp[adj_key] = sp.vstack(batch_connectivities)

adata_query.obs[mapping_entity_key] = "query"


In [ ]:
batch_connectivities = []
len_before_batch = 0

adata_query.obs[mapping_entity_key] = "query"

In [ ]:
adata_query.write(ADATA_PATH + ".svg.query")  
print(ADATA_PATH + ".svg.query")


# this adata can now be used as input for the ref-query tutorial for
nichecompass
https://nichecompass.readthedocs.io/en/latest/tutorials/notebooks/mouse_cns_spatial_reference_mapping.html

we apply this in a python job in ../job_scripts/nichecompass_ref_mapping2.py


In [ ]:
STOP

# Results

In [ ]:
PATH_RES = '/lustre/scratch124/cellgen/haniffa/projects/developmental_fibroblasts/nobackup_output/nichecompasss/nichecompass/artifacts/spatial_reference_mapping/20251201_091037XeniumTUTORIAL__REFQ/model/' 


In [ ]:
import os
os.listdir(PATH_RES + "reference")

In [ ]:
PATH_RES + "reference"

In [ ]:
import scanpy as sc
adata=sc.read_h5ad ( PATH_RES + "reference_query/adata.h5ad")
adata.obs["batch_nc"].value_counts()

In [ ]:
# import pickle

# outpath = "/nfs/team298/ls34/skin_niche19.pkl"

# with open(outpath, "rb") as f:
#     md_loaded = pickle.load(f)

# adata.obs["niche19"] = adata.obs.index.map(md_loaded)
# adata.obs["niche19"]=adata.obs["niche19"].fillna("QUERY_DATA")

In [ ]:
sc.settings.set_figure_params(dpi_save=300, facecolor="white", frameon=False, figsize=(20,20))

sc.pl.umap(
    adata,
    color=[
        "batch_nc",
    ],
    #legend_loc="on data",
    s=10,
    legend_fontsize=54,
    legend_fontoutline=2,
    show=False,
    #palette=colorsmap,
    edgecolor='black',
    linewidth=0.01,
    title='',
    palette=['lightblue', 'orange']
#   save=f"{x}_PAGA.pdf"
)

In [ ]:
colors_new2 = {
        'Epidermis_basal': "#4a538f",# "#3489f1", #"#000080", #"#f0f8ff", #"#007cfe",#'#4f59a1',  # dark blue
    "Epidermis basal":"#3489f1",# "#000080", #"#f0f8ff", #"#000080",
    
                'Epidermis_mid': '#F0F8FF', # #e3e7eb - GREY
  #  "Epidermis_late": "#4682B4", #"#A52A2A",          # pink
            'Epidermis_late':"#2a2f5c",#"#2a2f5c" ,# "#2e2e2e",#'#b1dae6',  # light blue

   'Epidermis basal_cycling': "#007cfe",  #  e2d8dd   "#505aa1",#"#c590a5" , # '#e0c5d2',
'Epidermis_basal_cycling': "#007cfe", 
'EpidermisInflamm_mid':"#f6e3ec",
        'EpidermisInflamm_late': "#800020",
    
    
    # 'Epidermis_Inflammatory': '#f2e7ed',
        "Epidermis_APChi": "#ff1493",
    'Papillary_dermis':"#FFFF00", # "#f6d17a", # yellow
    
  #  'Perivascular':  "#845DAE", #  '#fedbe7' PINKSIH,  #'#f8daeb',  # darkish red
    'Small blood vessel': "#d73435", #'#fe78bb',  # bright red
    'Sweat_gland': '#008080',  # dark grey
        'Sweat_gland_channel': "#adf7e8", #'#008080',  # original - 40E0D0
    'Muscle': "#f371af",  # very dark red
    'Adipocyte+vessels': "#B8860B", #'#FF4500',  # very dark orange
     'Hypodermis': "#FFFFF0", #"#D2B48C",#' # yello  # white = '#ffffff',
    'Larger blood vessel': "#660000",#"#8B0000", #'#FF6347',  # coral red
    'Perineural': '#0A0A0A',  # almost black
   # 'Plasma+pDC': '#808080',  # bright turquoise
    #'Sweat gland channel': '#000000',  # black
    #'KC_immunecell': '#FF1493',  # dark pink
    
   # 'Epidermis_late': '#b1dae6',
    "Nonspecific":"#D3D3D3",
     'Reticular_dermis': '#d0e1f2',

 'Large_BV': '#660000',
 'Small_BV': "#c43f40",#'#D96B6B',
     'Tzone-like': "#845DAE",# '#F4D1A1',

 'Sebaceous_gland': "#FFD9A3",      #  "#FFB347",#"#94cb72",#'#e28743',
"Sebaceous_immune": "#ff6145", 
 #'Plasma_cell_niche': "#1F51FF", #"#FFB300",#"#40E0D0",#'#ff5e00',
   # 'Plasma_cell_rich':  "#8080B2", #"
"Plasma_cell_rich": "#04D9FF",
    'HF_Perineural': '#00FF00',
    "OuterHF": "#006600",# 
 'Sebaceous duct': '#e28743',


"Lymphoid Tzone-like": "#825bac",
    



     #007FFF
    "Reticular_dermis_LE_rich" : "#6A8ED8",
        "Reticular_dermis_LErich" : "#6A8ED8",
    "Small_BV_Trich": "#FFB6C1",          # neon orange
    "Tzone-like_Theavy": "#B19CD9",       # light purple
    #"Reticular dermis_F2/F3hi": "#35476C", #"#FF5F1F",# light blue
    "HF_outer": "#90c670",                # very dark green
    "VenuleMuscle": "#B34A7E",

    "Sebaceous_duct": "#FFA07A",          # light orange (salmony)
    "VenuleEndothelium": "#8B0000",       # dark red
    "HF_inner": "#90EE90",                 # light green
    "HF_TNN+COCH+hi": "#39FF14",          # neon green
    # "Peri_sweat_gland": "#d9f9f7",
    'HF_innermost':  "#D95D54",
  #  'Sebaceous_inner_immune':"#F6A85C",
    'Nonspecific/folded': "#F0F0F0",
 'nan': '#FAFAFA',
    "QUERY_DATA": '#F0F0F0'
    }


In [ ]:
"""
how do niches compare to what we had before
"""
sc.pl.umap(
    adata,
    color=[
        "niche19",
    ],
    legend_loc="on data",
    s=5,
    legend_fontsize=14,
    legend_fontoutline=2,
    show=False,
    #palette=colorsmap,
    edgecolor='black',
    linewidth=0.01,
    title='',
    palette=colors_new2,
#   save=f"{x}_PAGA.pdf"
)

In [ ]:
"""
how do niches compare to what we had before
"""
sc.pl.umap(
    adata,
    color=[
        "niche19",
    ],
   # legend_loc="on data",
    s=5,
    legend_fontsize=14,
    legend_fontoutline=2,
    show=False,
    #palette=colorsmap,
    edgecolor='black',
    linewidth=0.01,
    title='',
    palette=colors_new2,
#   save=f"{x}_PAGA.pdf"
)

In [ ]:
# adata.write( PATH_RES + "reference_query/adata.h5ad")
